# 04 · Weather and forecast features (US Embassy station, OpenAQ 8160)

**Goal:** download hourly weather once (Open-Meteo, ERA5 reanalysis), turn it into daily weather by local day, and build the feature table for next-day PM2.5 forecasting.

**No-leakage rule:** a row is the forecast issued at the end of local day D for day D+1. Every feature uses day D or earlier. The only exception is the labelled `*_tplus1` weather columns, which exist only for the upper-bound experiment (`features.include_future_weather: true`) and are excluded by `feature_columns()` unless `future=True`.

All logic lives in `src/uaq/weather.py` and `src/uaq/features.py`; this notebook only calls it.

In [ ]:
# Setup: works whether or not you ran `pip install -e .`
import sys, pathlib, json
sys.path.insert(0, str(pathlib.Path.cwd().resolve().parent / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

from uaq.config import load_config, ROOT
from uaq import weather, features
from uaq.io import load_hourly
from uaq.daily import to_daily
from uaq.periods import assign_split, in_months
cfg = load_config()
pd.set_option("display.max_columns", 40); pd.set_option("display.width", 200)
%matplotlib inline
print("Project root:", ROOT)

## 1 · Download once
Requests one calendar year at a time in UTC (Mongolia used daylight saving time in 2015–2016) and converts to local time. Skipped if the file already exists.

In [ ]:
path = weather.download(cfg)
wh = weather.load_hourly(cfg)
print(f"{len(wh):,} hourly rows, {wh['ts_local'].min()} → {wh['ts_local'].max()}")
print(json.dumps(json.loads(path.with_name(path.stem + ".meta.json").read_text()), indent=2))

In [ ]:
wd = weather.to_daily(wh)
display(wd.describe().T.round(2))

### BLH availability rule (fixed before any model is trained)
Keep `w_blh_mean`, `w_blh_night` and `w_blh_night_change` only if each BLH column is present on ≥ 95% of smog-season days in **every** split (train, val, test). Otherwise they go in `features.drop_columns` and `feature_columns()` excludes them.

In [ ]:
fsp = cfg["forecast"]["periods"]["split"]
d = wd.assign(split=assign_split(wd["date"], fsp).values, smog=in_months(wd["date"], cfg["smog_months"]).values)
d = d[d["smog"] & d["split"].isin(["train", "val", "test"])]
blh = [c for c in ("blh_mean", "blh_night") if c in wd]
share = d.groupby("split")[blh].agg(lambda s: s.notna().mean()).reindex(["train", "val", "test"]) if blh else pd.DataFrame()
display(share.round(3).rename_axis("non-missing share, smog-season days"))
keep_blh = bool(blh) and bool((share >= 0.95).all().all())
blh_feats = ["w_blh_mean", "w_blh_night", "w_blh_night_change"]
drop = cfg["features"].get("drop_columns", [])
print("Rule outcome:", "KEEP the BLH features" if keep_blh else "DROP the BLH features (some split < 95%)")
assert keep_blh != set(blh_feats) <= set(drop), "features.drop_columns in the config disagrees with the rule: update it"
print("features.drop_columns:", drop)

## 2 · Weather vs PM2.5, training smog-season days only
Only **training** days (Nov–Feb, valid daily means) are looked at here. Exploring validation or test days would let what we see leak into feature and model choices.

In [ ]:
station = str(cfg["forecast"]["input"]["location_ids"][0])
pm_hourly = load_hourly(cfg)
pm_hourly = pm_hourly[pm_hourly["station_id"] == station]
pm_daily = to_daily(pm_hourly, cfg["quality"]["min_hours_per_day"])

m = pm_daily.merge(wd, on="date", how="inner")
train = ((assign_split(m["date"], cfg["forecast"]["periods"]["split"]) == "train").values
         & in_months(m["date"], cfg["smog_months"]).values & m["valid"].values)
tr = m[train]
wcols = [c for c in wd.columns if c not in ("date", "n_hours")]
rho = tr[wcols].corrwith(tr["pm25_daily"], method="spearman").sort_values()
print(f"{len(tr)} training smog-season days")
display(rho.round(2).to_frame("Spearman ρ with same-day PM2.5"))

In [ ]:
panel = [c for c in ("temp_min", "wind_mean", "blh_night", "rh_mean") if c in tr]
fig, axes = plt.subplots(1, len(panel), figsize=(4 * len(panel), 3.5), sharey=True)
for ax, c in zip(np.atleast_1d(axes), panel):
    ax.scatter(tr[c], tr["pm25_daily"], s=6, alpha=0.4)
    ax.set_xlabel(c); ax.set_title(f"ρ = {rho[c]:.2f}")
np.atleast_1d(axes)[0].set_ylabel("daily mean PM2.5 (µg/m³)")
fig.tight_layout(); plt.show()

## 3 · Build the feature table
One row per issue day D for the forecasting station.

In [ ]:
feat = features.build(pm_daily, pm_hourly, cfg, wd)
with_w = features.feature_columns(feat, drop=drop)
without_w = features.feature_columns(feat, weather=False, drop=drop)
print(f"{len(feat):,} issue days; {len(with_w)} features with weather, {len(without_w)} without")
display(feat.head())

## 4 · Usable rows per split (smog-season targets)

**Fair weather ablation:** the with-weather and no-weather models are trained and scored on the **same rows**, those complete for the with-weather feature set (`ablation_rows` below). The no-weather model just ignores the `w_*` columns, so any difference comes from the features, not from different days.

In [ ]:
s = feat[feat["smog"]]
usable = pd.DataFrame({
    "issue days": s.groupby("split").size(),
    "target present": s[s["y"].notna()].groupby("split").size(),
    "complete with weather": features.complete_rows(s, with_w).groupby("split").size(),
    "complete without weather": features.complete_rows(s, without_w).groupby("split").size(),
}).reindex(["train", "val", "test", "outside"]).fillna(0).astype(int)
ablation_rows = features.complete_rows(feat, with_w)          # use for BOTH models in the ablation
usable["ablation rows (both models)"] = ablation_rows[ablation_rows["smog"]].groupby("split").size()
usable = usable.fillna(0).astype(int)
display(usable)

In [ ]:
seq_cols = (["pm_lag0", "pm_evening", "pm_max_hourly", "doy_sin", "doy_cos"]
            + [c for c in with_w if c.startswith("w_") and not c.endswith("_change")])
T = cfg["forecast"]["history_days"]
X, y, issue = features.make_sequences(feat, seq_cols, T)
print("X:", X.shape, " y:", y.shape)
meta = feat.set_index("date").loc[issue, ["split", "smog"]]
display(pd.crosstab(meta["split"], meta["smog"], margins=True))

## 5 · Missing values per feature (training, smog season)

In [ ]:
tr_smog = feat[(feat["split"] == "train") & feat["smog"]]
display(tr_smog[with_w].isna().mean().sort_values(ascending=False).round(3).to_frame("share missing"))

## 6 · Save
`data/processed/` is git-ignored.

In [ ]:
out = ROOT / "data" / "processed" / f"features_{station}.csv"
out.parent.mkdir(parents=True, exist_ok=True)
feat.to_csv(out, index=False)
print("Saved", out)